# Wikiquote Multipage Scraper
Crawls Italian Wikiquote's "Tutte le pagine" (_"All pages"_) index page by page, visits every linked article, and extracts cleaned quotes into a single JSON file.

## Setup

In [1]:
from bs4 import BeautifulSoup
import requests
import re
import json
import time

In [2]:
# Add headers to mimic a normal browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/129.0.0.0 Safari/537.36"
}

## Config

In [3]:
base_url = "https://it.wikiquote.org"
current_url = "https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Tot%C3%B2%2C+Peppino+e+la+dolce+vita"
to_remove = {'/wiki/%25s', 'Wikipedia', 'Wikinotizie', 'Commons', 'Altri progetti'}
IGNORE = ["Indice", "Note", "Voci correlate", "Altri progetti", "Bibliografia"]
patterns = [r"\[.+?\]", r"\(.+?\)"]

all_data = []

## Scraping
Walks the alphabetical page index, following "Pagina successiva" (_"Next Page"_) until there are no more pages, and scrapes quotes from every article found along the way.

In [4]:
i = 0
while i <= 20:

    # Fetch and parse the page
    response = requests.get(current_url, headers=headers)
    time.sleep(1)
    soup = BeautifulSoup(response.text, "html.parser")

    print(current_url)

    # Find the "Pagina successiva" link
    next_tag = soup.find(string=lambda s: s and "Pagina successiva" in s)

    # --------- GET ALL LINKS IN CURRENT PAGE ------------

    all_pages = soup.find(class_="mw-allpages-chunk")

    all_links = []

    for link in all_pages.find_all('a'):
        all_links.append(link.get('href'))

    all_links = [l for l in all_links if l.strip() not in to_remove]

    # ----------------------------------------------------

    # --------- QUOTE EXTRACTION -------------------------

    for rel_link in all_links:
        url = base_url + rel_link

        response = requests.get(url, headers=headers)
        time.sleep(1)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.string

        quotes = []

        # --- Extract quotes from <p> ---
        for p in soup.find_all("p"):
            text = p.get_text(separator=" ", strip=True)

            # Skip ignored or small-tag paragraphs
            if p.find("small") or text in IGNORE:
                continue

            # Skip paragraphs starting with "Etichetta:"
            if text.startswith("Etichetta:"):
                continue

            # Check if paragraph starts with "Citazioni su..."
            if re.match(r"^Citazioni su.+?", text):
                ul = p.find_next("ul")
                if ul:
                    for li in ul.find_all("li"):
                        quotes.append(li.get_text(separator=" ", strip=True))
                continue  # Skip adding the "Citazioni su..." paragraph itself

        # Otherwise, add the normal paragraph text
        quotes.append(text)

        # --- Extract quotes from <h3> blocks ---
        for h3 in soup.find_all("h3"):
            if h3.get_text() not in IGNORE:
                if h3.i:
                    quotes.append(h3.i.get_text())

                ul = h3.find_next("ul")
                next_h3 = h3.find_next("h3")
                next_h2 = h3.find_next("h2")

                if (next_h3 and next_h2):
                    if (ul != next_h3.find_next("ul") and ul != next_h2.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (next_h3 and not next_h2):
                    if (ul != next_h3.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (not next_h3 and next_h2):
                    if (ul != next_h2.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (not next_h3 and not next_h2):
                    for li in ul.find_all("li"):
                        quotes.append(li.get_text())

        # --- Extract quotes from <h2> blocks ---
        for h2 in soup.find_all("h2"):
            if h2.get_text() not in IGNORE:
                ul = h2.find_next("ul")
                next_h3 = h2.find_next("h3")
                next_h2 = h2.find_next("h2")

                if (next_h3 and next_h2):
                    if (ul != next_h3.find_next("ul") and ul != next_h2.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (next_h3 and not next_h2):
                    if (ul != next_h3.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (not next_h3 and next_h2):
                    if (ul != next_h2.find_next("ul")):
                        for li in ul.find_all("li"):
                            quotes.append(li.get_text())
                elif (not next_h3 and not next_h2):
                    for li in ul.find_all("li"):
                        quotes.append(li.get_text())

        # --- Clean up quotes ---

        cleaned_quotes = []
        for q in quotes:
            new_q = q
            for pattern in patterns:
                new_q = re.sub(pattern, "", new_q)
            new_q = new_q.strip()
            if new_q and new_q not in to_remove:
                cleaned_quotes.append(new_q)

        # Save page results
        page_data = {"page": url, "title": title, "quotes": cleaned_quotes}
        all_data.append(page_data)

        # ----------------------------------------------------

    if not next_tag:
        print("No more pages.")
        break  # stop when there's no next page

    link_tag = next_tag.find_parent("a")
    if not link_tag or not link_tag.get("href"):
        print("No valid 'Pagina successiva' link found.")
        break

    # Build full URL for next iteration
    next_url = requests.compat.urljoin(base_url, link_tag["href"])
    current_url = next_url

    i += 1

https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Tot%C3%B2%2C+Peppino+e+la+dolce+vita
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Trentacinquenne
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Turgenev
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Ugo+da+Prato
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Un+lungo+fatale+inseguimento+d%27amore
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Unire
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Val+Poschiavo
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Varzo
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Verlyn+Flieger
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Vienna
https://it.wikiquote.org/w/index.php?title=Speciale:TutteLePagine&from=Violator
https://it.wikiquote.org/w/index.php?title=Speci

## Save Output

In [5]:
# Save as JSON
output_path = "/your/output/path.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)